In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BRONZE = "abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/"
silver_path = "abfss://silver@logisticdatalakestorage.dfs.core.windows.net/openmeteo/"

### Build City Reference from config.json

In [0]:
df_cities_raw = spark.read.option('multiline', 'true').json(BRONZE + 'config/config.json')

df_city_ref = df_cities_raw \
    .select(explode(col('cities')).alias('c')) \
    .select(
        initcap(col('c.name')).alias('mapped_city'),
        upper(col('c.country')).alias('mapped_country'),
        col('c.lat').alias('c_lat'),
        col('c.lon').alias('c_lon')
    )

print(f'City reference rows: {df_city_ref.count()}')
df_city_ref.display()

City reference rows: 10


mapped_city,mapped_country,c_lat,c_lon
Mumbai,IN,19.076,72.8777
Berlin,DE,52.52,13.405
London,GB,51.5074,-0.1278
New York,US,40.7128,-74.006
Bangkok,TH,13.7563,100.5018
Dubai,AE,25.2048,55.2708
Singapore,SG,1.3521,103.8198
Sydney,AU,-33.8688,151.2093
Paris,FR,48.8566,2.3522
Sao Paulo,BR,-23.5505,-46.6333


### Read Bronze JSON files

In [0]:
df = spark.read.json(BRONZE + 'openmeteo/')
df.limit(2).display()

elevation generationtime_ms hourly hourly_units latitude longitude timezone timezone_abbreviation utc_offset_seconds year month day hour 6.0 0.25284290313720703 List(List(37, 36, 35, 33, 28, 21, 16, 13, 12, 10, 7, 3, 0, 0, 0, 0, 0, 0, 0, 1, 2, 4, 9, 14, 18, 18, 15, 12, 9, 5, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 6, 8, 10, 12, 13, 13, 12, 10, 8, 6, 5, 5, 4, 3, 1, 0, 0, 1, 2, 4, 6, 8, 10, 12, 14, 17, 21, 24, 27, 30, 31, 29, 26, 22, 18, 15, 12, 11, 10, 10, 9, 8, 8, 11, 16, 20, 22, 23, 25, 28, 32, 37, 44, 53, 57, 54, 47, 41, 38, 35, 35, 39, 44, 47, 43, 37, 31, 28, 25, 24, 24, 25, 27, 31, 36, 39, 39, 38, 39, 46, 56, 63, 65, 64, 63, 62, 59, 55, 47, 36, 27, 20, 14, 12, 16, 24, 30, 33, 34, 35, 37, 39, 41, 43, 46, 49, 53, 57, 61, 64, 67, 67, 62, 55, 47, 39, 32, 27, 28, 32, 36, 41, 46), List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(29.0, 29.2, 30.4, 31.8, 33.2, 34.0, 34.3, 34.5, 34.4, 34.2, 34.0, 33.4, 32.7, 32.0, 31.4, 31.1, 30.9, 30.8, 30.6, 30.5, 30.2, 30.0, 29.9, 29.8, 29.5, 29.5, 30.5, 31.5, 32.4, 33.1, 33.8, 33.8, 33.5, 33.2, 32.8, 32.4, 31.8, 31.1, 30.7, 30.6, 30.5, 30.4, 30.4, 30.3, 30.2, 30.1, 29.9, 29.9, 29.9, 30.1, 30.9, 31.5, 32.0, 32.3, 33.1, 33.2, 33.4, 33.2, 33.1, 32.7, 32.1, 31.6, 31.1, 30.8, 30.8, 30.5, 30.4, 30.5, 30.2, 30.3, 30.2, 30.2, 30.0, 30.1, 30.5, 31.4, 32.0, 32.8, 33.2, 33.7, 33.8, 33.8, 33.5, 32.9, 32.2, 31.4, 30.9, 30.8, 30.8, 30.6, 30.5, 30.5, 30.5, 30.4, 30.1, 30.0, 30.0, 30.3, 30.9, 31.5, 32.2, 32.9, 33.5, 33.7, 33.6, 33.3, 32.8, 32.0, 31.3, 30.8, 30.2, 29.8, 29.5, 29.2, 29.1, 29.0, 29.0, 28.9, 28.5, 28.1, 28.0, 28.4, 29.1, 30.0, 30.9, 31.8, 32.5, 32.7, 32.5, 32.3, 32.2, 32.0, 31.7, 31.3, 30.9, 30.5, 30.2, 30.0, 29.9, 29.6, 29.4, 29.0, 28.2, 28.1, 28.2, 28.7, 29.5, 30.5, 31.5, 32.4, 32.9, 33.0, 33.0, 32.8, 32.5, 32.1, 31.9, 31.6, 31.4, 31.1, 30.9, 30.6, 30.4, 30.1, 29.7, 29.3, 29.0, 28.8), List(2026-05-27T00:00, 2026-05-27T01:00, 2026-05-27T02:00, 2026-05-27T03:00, 2026-05-27T04:00, 2026-05-27T05:00, 2026-05-27T06:00, 2026-05-27T07:00, 2026-05-27T08:00, 2026-05-27T09:00, 2026-05-27T10:00, 2026-05-27T11:00, 2026-05-27T12:00, 2026-05-27T13:00, 2026-05-27T14:00, 2026-05-27T15:00, 2026-05-27T16:00, 2026-05-27T17:00, 2026-05-27T18:00, 2026-05-27T19:00, 2026-05-27T20:00, 2026-05-27T21:00, 2026-05-27T22:00, 2026-05-27T23:00, 2026-05-28T00:00, 2026-05-28T01:00, 2026-05-28T02:00, 2026-05-

In [0]:
print(df.count())

10


### zip the object array values to array

In [0]:
df2 = df.withColumn('weather', arrays_zip(
        "hourly.time",
        "hourly.temperature_2m",
        "hourly.rain",
        "hourly.wind_speed_10m",
        "hourly.precipitation_probability",
        "hourly.visibility",
        "hourly.weather_code"
))

In [0]:
df2.limit(2).display()

elevation generationtime_ms hourly hourly_units latitude longitude timezone timezone_abbreviation utc_offset_seconds year month day hour weather 6.0 0.25284290313720703 List(List(37, 36, 35, 33, 28, 21, 16, 13, 12, 10, 7, 3, 0, 0, 0, 0, 0, 0, 0, 1, 2, 4, 9, 14, 18, 18, 15, 12, 9, 5, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 6, 8, 10, 12, 13, 13, 12, 10, 8, 6, 5, 5, 4, 3, 1, 0, 0, 1, 2, 4, 6, 8, 10, 12, 14, 17, 21, 24, 27, 30, 31, 29, 26, 22, 18, 15, 12, 11, 10, 10, 9, 8, 8, 11, 16, 20, 22, 23, 25, 28, 32, 37, 44, 53, 57, 54, 47, 41, 38, 35, 35, 39, 44, 47, 43, 37, 31, 28, 25, 24, 24, 25, 27, 31, 36, 39, 39, 38, 39, 46, 56, 63, 65, 64, 63, 62, 59, 55, 47, 36, 27, 20, 14, 12, 16, 24, 30, 33, 34, 35, 37, 39, 41, 43, 46, 49, 53, 57, 61, 64, 67, 67, 62, 55, 47, 39, 32, 27, 28, 32, 36, 41, 46), List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(29.0, 29.2, 30.4, 31.8, 33.2, 34.0, 34.3, 34.5, 34.4, 34.2, 34.0, 33.4, 32.7, 32.0, 31.4, 31.1, 30.9, 30.8, 30.6, 30.5, 30.2, 30.0, 29.9, 29.8, 29.5, 29.5, 30.5, 31.5, 32.4, 33.1, 33.8, 33.8, 33.5, 33.2, 32.8, 32.4, 31.8, 31.1, 30.7, 30.6, 30.5, 30.4, 30.4, 30.3, 30.2, 30.1, 29.9, 29.9, 29.9, 30.1, 30.9, 31.5, 32.0, 32.3, 33.1, 33.2, 33.4, 33.2, 33.1, 32.7, 32.1, 31.6, 31.1, 30.8, 30.8, 30.5, 30.4, 30.5, 30.2, 30.3, 30.2, 30.2, 30.0, 30.1, 30.5, 31.4, 32.0, 32.8, 33.2, 33.7, 33.8, 33.8, 33.5, 32.9, 32.2, 31.4, 30.9, 30.8, 30.8, 30.6, 30.5, 30.5, 30.5, 30.4, 30.1, 30.0, 30.0, 30.3, 30.9, 31.5, 32.2, 32.9, 33.5, 33.7, 33.6, 33.3, 32.8, 32.0, 31.3, 30.8, 30.2, 29.8, 29.5, 29.2, 29.1, 29.0, 29.0, 28.9, 28.5, 28.1, 28.0, 28.4, 29.1, 30.0, 30.9, 31.8, 32.5, 32.7, 32.5, 32.3, 32.2, 32.0, 31.7, 31.3, 30.9, 30.5, 30.2, 30.0, 29.9, 29.6, 29.4, 29.0, 28.2, 28.1, 28.2, 28.7, 29.5, 30.5, 31.5, 32.4, 32.9, 33.0, 33.0, 32.8, 32.5, 32.1, 31.9, 31.6, 31.4, 31.1, 30.9, 30.6, 30.4, 30.1, 29.7, 29.3, 29.0, 28.8), List(2026-05-27T00:00, 2026-05-27T01:00, 2026-05-27T02:00, 2026-05-27T03:00, 2026-05-27T04:00, 2026-05-27T05:00, 2026-05-27T06:00, 2026-05-27T07:00, 2026-05-27T08:00, 2026-05-27T09:00, 2026-05-27T10:00, 2026-05-27T11:00, 2026-05-27T12:00, 2026-05-27T13:00, 2026-05-27T14:00, 2026-05-27T15:00, 2026-05-27T16:00, 2026-05-27T17:00, 2026-05-27T18:00, 2026-05-27T19:00, 2026-05-27T20:00, 2026-05-27T21:00, 2026-05-27T22:00, 2026-05-27T23:00, 2026-05-28T00:00, 2026-05-28T01:00, 2026-05-28T02:00, 

### Explode the array

In [0]:
df3 = df2.withColumn("weather", explode("weather"))

In [0]:
df3 = df3.drop("hourly", "hourly_units")

In [0]:
df3.limit(2).display()

elevation,generationtime_ms,latitude,longitude,timezone,timezone_abbreviation,utc_offset_seconds,year,month,day,hour,weather
6.0,0.25284290313720703,19.086115,72.85291,GMT,GMT,0,2026,5,27,12,"List(2026-05-27T00:00, 29.0, 0.0, 7.2, 37, 9120.0, 95)"
6.0,0.25284290313720703,19.086115,72.85291,GMT,GMT,0,2026,5,27,12,"List(2026-05-27T01:00, 29.2, 0.0, 6.5, 36, 9320.0, 95)"


### flatten The Json

In [0]:
df_silver = df3.select(
    '*',
    col('weather.time').alias('weather_datetime'),
    col('weather.temperature_2m').alias('temperature'),
    col('weather.rain').alias('rain'),
    col('weather.precipitation_probability').alias('precipitation_probability'),
    col('weather.visibility').alias('visibility'),
    col('weather.wind_speed_10m').alias('wind_speed_10m'),
    col('weather.weather_code').alias('weather_code')
).drop('timezone', 'timezone_abbreviation', 'utc_offset_seconds','weather')

In [0]:
df_silver.limit(2).display()

elevation,generationtime_ms,latitude,longitude,year,month,day,hour,weather_datetime,temperature,rain,precipitation_probability,visibility,wind_speed_10m,weather_code
6.0,0.25284290313720703,19.086115,72.85291,2026,5,27,12,2026-05-27T00:00,29.0,0.0,37,9120.0,7.2,95
6.0,0.25284290313720703,19.086115,72.85291,2026,5,27,12,2026-05-27T01:00,29.2,0.0,36,9320.0,6.5,95


### Map lat/lon → Correct City Name

In [0]:
window_city = Window.partitionBy('latitude', 'longitude', 'weather_datetime').orderBy('_dist')

df_silver = df_silver \
    .crossJoin(broadcast(df_city_ref)) \
    .withColumn('_lat_diff', abs(col('latitude') - col('c_lat'))) \
    .withColumn('_lon_diff', abs(col('longitude') - col('c_lon'))) \
    .withColumn('_dist', col('_lat_diff') + col('_lon_diff')) \
    .filter(col('_dist') < 1.0) \
    .withColumn('_rn', row_number().over(window_city)) \
    .filter(col('_rn') == 1) \
    .drop('_lat_diff', '_lon_diff', '_dist', '_rn', 'c_lat', 'c_lon') \
    .withColumnRenamed('mapped_city', 'city') \
    .withColumnRenamed('mapped_country', 'country_code')

print(f'Rows after city mapping: {df_silver.count()}')
df_silver.select('city', 'country_code', 'weather_datetime').limit(5).display()

Rows after city mapping: 1680


city,country_code,weather_datetime
Sydney,AU,2026-05-27T00:00
Sydney,AU,2026-05-27T01:00
Sydney,AU,2026-05-27T02:00
Sydney,AU,2026-05-27T03:00
Sydney,AU,2026-05-27T04:00


### Create Join_time

In [0]:
df_silver = df_silver \
    .withColumn('event_time', to_timestamp(col('weather_datetime'), "yyyy-MM-dd'T'HH:mm")) \
    .withColumn('join_time', date_trunc('hour', col('event_time'))) \
    .drop('weather_datetime')

print('Distinct cities:')
df_silver.select('city', 'country_code').distinct().orderBy('city').show()
print(f'Total rows: {df_silver.count()}')

Distinct cities:
+---------+------------+
|     city|country_code|
+---------+------------+
|  Bangkok|          TH|
|   Berlin|          DE|
|    Dubai|          AE|
|   London|          GB|
|   Mumbai|          IN|
| New York|          US|
|    Paris|          FR|
|Sao Paulo|          BR|
|Singapore|          SG|
|   Sydney|          AU|
+---------+------------+

Total rows: 1680


In [0]:
df_silver.limit(5).display()

elevation,generationtime_ms,latitude,longitude,year,month,day,hour,temperature,rain,precipitation_probability,visibility,wind_speed_10m,weather_code,city,country_code,event_time,join_time
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,17.4,0.0,0,640.0,5.4,3,Sydney,AU,2026-05-27T00:00:00.000Z,2026-05-27T00:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,19.0,0.0,0,3620.0,4.9,3,Sydney,AU,2026-05-27T01:00:00.000Z,2026-05-27T01:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.1,0.0,2,12280.0,2.4,2,Sydney,AU,2026-05-27T02:00:00.000Z,2026-05-27T02:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.8,0.0,8,13340.0,2.6,1,Sydney,AU,2026-05-27T03:00:00.000Z,2026-05-27T03:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.9,0.0,23,12260.0,4.8,1,Sydney,AU,2026-05-27T04:00:00.000Z,2026-05-27T04:00:00.000Z


### Write to Silver Delta Lake

In [0]:
df_silver.write.format('delta').mode('overwrite').partitionBy("year", "month", "day").save(silver_path)